<a href="https://colab.research.google.com/github/shivashankarb2006/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shivashankarb2006/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Method Choice and Why

I chose Random Forest Classification for the Content Opportunity Scoring lane.

The task is to identify content pages that are potential optimization opportunities. Random Forest can combine several search-performance and content signals and capture non-linear relationships between them.

It is also useful for interpretation because feature importance can show which signals contribute most to the model's decisions.

The model is used for decision support and ranking, not as proof of causation or as a prediction of Google's ranking algorithm.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Split Design

I use a GroupShuffleSplit based on client_hash_id.

Pages from the same client are kept in the same train or test group. This reduces the risk that the model learns client-specific patterns and then appears to perform well simply because similar pages from the same client occur in both sets.

The split is therefore intended to provide a more honest estimate of whether the model can generalize across clients.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [2]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn pandas

In [3]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

print("DuckDB connected successfully.")

DuckDB connected successfully.


In [4]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions ELSE 0
                END
            ) AS imp_last30,

            SUM(
                CASE
                    WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions ELSE 0
                END
            ) AS imp_prev30,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_clicks ELSE 0
                END
            ) AS clk_last30,

            AVG(
                CASE
                    WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_avg_position
                END
            ) AS pos_prev30

        FROM {TABLES['fact_daily']} f, bounds b

        WHERE f.report_date > b.end_d - INTERVAL 90 DAY

        GROUP BY 1, 2

        HAVING imp_prev30 >= 50
    )

    SELECT *
    FROM windowed
""").df()

print("Feature rows:", len(features))
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 155903


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_prev30
0,client_e547b89c05043229,content_25dfa3e39bc37247,216.0,1060.0,1.0,6.829891
1,client_e547b89c05043229,content_4f6ed7741dfd65e4,9.0,98.0,0.0,18.628244
2,client_e547b89c05043229,content_bab118937886d46a,110.0,359.0,0.0,22.740435
3,client_e547b89c05043229,content_7206e9a3ceefb37a,279.0,226.0,1.0,23.324573
4,client_e547b89c05043229,content_0587243c78e9468a,48.0,180.0,0.0,35.190421


In [5]:
qsignals = con.sql(f"""
    SELECT
        content_hash_id,

        ANY_VALUE(content_visible_query_count)
            AS visible_queries,

        ANY_VALUE(rare_impressions_share)
            AS rare_share,

        ANY_VALUE(anonymized_impressions_share)
            AS anon_share,

        MAX(impressions_90d)
            AS top_query_impressions,

        SUM(impressions_90d)
            AS kept_impressions

    FROM {TABLES['fact_query_90d']}

    GROUP BY content_hash_id
""").df()

qsignals["top_query_share"] = (
    qsignals["top_query_impressions"] /
    qsignals["kept_impressions"]
)

qsignals["query_diversity"] = (
    1 - qsignals["top_query_share"]
)

data = features.merge(
    qsignals,
    on="content_hash_id",
    how="left"
)

print("Modeling rows:", len(data))
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 155903


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_prev30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share,query_diversity
0,client_e547b89c05043229,content_25dfa3e39bc37247,216.0,1060.0,1.0,6.829891,5.0,0.038401,0.897335,34.0,82.0,0.414634,0.585366
1,client_e547b89c05043229,content_4f6ed7741dfd65e4,9.0,98.0,0.0,18.628244,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,client_e547b89c05043229,content_bab118937886d46a,110.0,359.0,0.0,22.740435,1.0,0.168443,0.528785,142.0,142.0,1.000000,0.000000
3,client_e547b89c05043229,content_7206e9a3ceefb37a,279.0,226.0,1.0,23.324573,1.0,0.219802,0.748515,16.0,16.0,1.000000,0.000000
4,client_e547b89c05043229,content_0587243c78e9468a,48.0,180.0,0.0,35.190421,1.0,0.118421,0.763158,27.0,27.0,1.000000,0.000000


In [6]:
data["is_declining"] = (
    data["imp_last30"] < 0.8 * data["imp_prev30"]
).astype(int)

feature_cols = [
    "imp_prev30",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share",
    "query_diversity"
]

model_data = data.dropna(
    subset=feature_cols + ["client_hash_id"]
).copy()

X = model_data[feature_cols]
y = model_data["is_declining"]
groups = model_data["client_hash_id"]

print("Model rows:", len(model_data))
print("Declining rate:", y.mean())

Model rows: 124268
Declining rate: 0.8673190201821869


In [9]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

Train rows: 108877
Test rows: 15391


In [10]:
baseline_label = y_train.mode().iloc[0]

baseline_pred = [baseline_label] * len(y_test)

baseline_metrics = {
    "accuracy": accuracy_score(y_test, baseline_pred),
    "precision": precision_score(
        y_test, baseline_pred, zero_division=0
    ),
    "recall": recall_score(
        y_test, baseline_pred, zero_division=0
    ),
    "f1": f1_score(
        y_test, baseline_pred, zero_division=0
    )
}

baseline_metrics

{'accuracy': 0.8546553180430122,
 'precision': 0.8546553180430122,
 'recall': 1.0,
 'f1': 0.9216325100718165}

In [11]:
model_metrics = {
    "accuracy": accuracy_score(y_test, pred),
    "precision": precision_score(
        y_test, pred, zero_division=0
    ),
    "recall": recall_score(
        y_test, pred, zero_division=0
    ),
    "f1": f1_score(
        y_test, pred, zero_division=0
    )
}

model_metrics

{'accuracy': 0.9298291209148204,
 'precision': 0.928034600113443,
 'recall': 0.9950585373270489,
 'f1': 0.9603786044464011}

In [14]:
comparison = pd.DataFrame(
    [baseline_metrics, model_metrics],
    index=["Baseline", "Random Forest"]
)

comparison

,accuracy,precision,recall,f1
Baseline,0.854655,0.854655,1.000000,0.921633
Random Forest,0.929829,0.928035,0.995059,0.960379


In [13]:
import pandas as pd

## Errors and Interpretation

The model should not be judged only by its overall accuracy.

I will inspect false positives and false negatives to understand where the model makes mistakes.

A false positive means that the model identifies a page as declining when the observed label does not indicate a decline.

A false negative means that the model misses a page that meets the decline definition.

Feature importance will be used as a directional interpretation of which signals the model relies on. Feature importance does not establish causality.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [15]:
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

importance

,feature,importance
3,anon_share,0.231380
0,imp_prev30,0.231235
2,rare_share,0.189230
1,visible_queries,0.126685
4,top_query_share,0.110884
5,query_diversity,0.110587


In [16]:
errors = model_data.iloc[test_idx].copy()

errors["actual"] = y_test.values
errors["predicted"] = pred

print("False positives:")
display(
    errors[
        (errors["actual"] == 0) &
        (errors["predicted"] == 1)
    ][feature_cols + ["actual", "predicted"]].head(10)
)

print("False negatives:")
display(
    errors[
        (errors["actual"] == 1) &
        (errors["predicted"] == 0)
    ][feature_cols + ["actual", "predicted"]].head(10)
)

False positives:


,imp_prev30,visible_queries,rare_share,anon_share,top_query_share,query_diversity,actual,predicted
12001,1133.0,3.0,0.011152,0.926064,0.750000,0.250000,0,1
12002,1400.0,6.0,0.006486,0.954304,0.255639,0.744361,0,1
12277,11376.0,46.0,0.008477,0.716878,0.356910,0.643090,0,1
12292,334.0,9.0,0.286834,0.344828,0.242553,0.757447,0,1
12462,5667.0,21.0,0.019608,0.861635,0.151869,0.848131,0,1
12671,3536.0,8.0,0.012436,0.824343,0.450476,0.549524,0,1
12884,2123.0,14.0,0.014328,0.478053,0.265681,0.734319,0,1
12913,1296.0,13.0,0.071220,0.758583,0.201717,0.798283,0,1
12925,1608.0,11.0,0.031825,0.749142,0.500000,0.500000,0,1
13057,301.0,4.0,0.087566,0.567426,0.451777,0.548223,0,1


False negatives:


,imp_prev30,visible_queries,rare_share,anon_share,top_query_share,query_diversity,actual,predicted
12798,594.0,4.0,0.046243,0.893642,0.346154,0.653846,1,0
13541,878.0,35.0,0.444937,0.085735,0.102362,0.897638,1,0
13592,104.0,5.0,0.467949,0.153846,0.288136,0.711864,1,0
13633,611.0,6.0,0.036503,0.865514,0.294118,0.705882,1,0
13650,51.0,3.0,0.253521,0.253521,0.400000,0.600000,1,0
15521,259.0,1.0,0.166282,0.810624,1.000000,0.000000,1,0
15874,227.0,1.0,0.181102,0.792651,1.000000,0.000000,1,0
15920,1972.0,16.0,0.365122,0.527356,0.243816,0.756184,1,0
16597,243.0,1.0,0.043624,0.922819,1.000000,0.000000,1,0
32887,2964.0,72.0,0.060935,0.276091,0.081551,0.918449,1,0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.